# 1. Import Libraries, Configuration Setup, and Load the Dataset

**Experiment:** `03_Full_aspects_aug`. Predict efficacy, safety, burden, cost separately, with three classes per aspect.

Input: review text plus condition, ingredients and ingredient count. ALBERT and BioBERT are trained and evaluated separately. Labels are training targets, not input features.

Follow the cells in order. Each model has its own tokenization, search and evaluation section, following `R2_Aspect_Seed_2025`. The existing DrugReview labels, loss, search settings and output files are retained.


## 1.1 Install \& Import Libraries

In [1]:
# Run before imports in a fresh Colab runtime. PyTorch is supplied by Colab.
%pip -q install transformers==4.57.6 scikit-learn pandas matplotlib seaborn nltk lightgbm statsmodels tqdm joblib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 175.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 153.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# No additional install is needed here; packages are installed in the cell above.


In [3]:
# No additional install is needed here; packages are installed in the cell above.


In [4]:
# No additional install is needed here; packages are installed in the cell above.


In [5]:
import os
import re
import gc
import json
import math
import random
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

import time
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

def aux_hash(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def aux_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

def aux_validate_config(cfg):
    aspects = tuple(cfg["aspects"])
    if len(set(aspects)) != len(aspects) or any(a not in AUX_ASPECTS for a in aspects):
        raise ValueError("Unknown or repeated aspect.")
    if cfg["target"] not in ("original", "holistic", "aspects"):
        raise ValueError("Unknown training target.")
    if cfg["target"] == "original" and aspects:
        raise ValueError("Original baseline has no auxiliary heads.")
    if cfg["target"] == "aspects" and aspects != AUX_ASPECTS:
        raise ValueError("Standalone full-aspects condition must predict all four aspects.")
    if not math.isfinite(cfg["aux_strength"]) or cfg["aux_strength"] < 0:
        raise ValueError("Auxiliary strength must be finite and non-negative.")
    if cfg["target"] == "holistic" and aspects and cfg["aux_strength"] <= 0:
        raise ValueError("Joint condition needs positive auxiliary strength.")
    if cfg["n_trials"] < 1 or cfg["batch_size"] < 1 or cfg["max_length"] < 1:
        raise ValueError("Training sizes must be positive.")
    if not cfg["backbones"] or any(b not in AUX_BACKBONES for b in cfg["backbones"]):
        raise ValueError("Select ALBERT and/or BioBERT.")


def require_finite_tensor(value, description):
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"Non-finite {description}; no checkpoint or prediction will be selected.")

def require_finite_model(model):
    for name, value in model.state_dict().items():
        if value.is_floating_point():
            require_finite_tensor(value, f"model state {name}")

def require_probabilities(values, n_classes):
    values = np.asarray(values)
    if (values.ndim != 2 or values.shape[1] != n_classes or not np.isfinite(values).all()
            or (values < 0).any() or (values > 1).any()
            or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5, rtol=1e-5)):
        raise ValueError("Invalid probability matrix.")
    return values

def checked_optimizer_step(loss, model, optimizer, scaler, max_norm, scheduler=None):
    require_finite_tensor(loss, "training loss")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm, error_if_nonfinite=not scaler.is_enabled())
    previous_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    updated = scaler.get_scale() >= previous_scale
    if updated and not torch.isfinite(grad_norm):
        raise FloatingPointError("Non-finite gradient norm without a skipped optimizer update.")
    if updated and scheduler is not None:
        scheduler.step()
    return updated


## 1.2 Seed \& Device

In [6]:
RUN_SEED = 2025
SEED = RUN_SEED

def aux_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def aux_worker_seed(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

aux_seed(RUN_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Search seed: {RUN_SEED} | Device: {DEVICE}")
# Trial i uses RUN_SEED + i; the saved data split is independent of training seeds.


Search seed: 2025 | Device: cuda


## 1.3 Configuration

In [7]:
PROJECT_ROOT = Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")
EXPERIMENT_FOLDER = '03_Full_aspects_aug'
TRAINING_TARGET = 'aspects'
ASPECT_NAMES = ['efficacy', 'safety', 'burden', 'cost']
USE_METADATA = True

# Existing DrugReview settings; these are unchanged by the notebook reorganization.
N_ITERATIONS = 10
MAX_TOKEN_LENGTH = 200
BATCH_SIZE = 128
PATIENCE = 3
NUM_WORKERS = 2
AUXILIARY_STRENGTH = 1.0
BACKBONES = ['albert', 'biobert']

AUX_CONFIG = {
    "folder": EXPERIMENT_FOLDER, "target": TRAINING_TARGET,
    "aspects": ASPECT_NAMES, "augmentation": USE_METADATA,
    "aux_strength": AUXILIARY_STRENGTH, "n_trials": N_ITERATIONS,
    "seed": RUN_SEED, "max_length": MAX_TOKEN_LENGTH,
    "batch_size": BATCH_SIZE, "patience": PATIENCE,
    "num_workers": NUM_WORKERS, "backbones": BACKBONES,
}

AUX_VERSION = "aspect_auxiliary_v2_3class"
AUX_ASPECTS = ("efficacy", "safety", "burden", "cost")
# The immutable GPT release keeps all five source states. Only supervision is collapsed.
AUX_SOURCE_STATES = ("NEGATIVE", "POSITIVE", "MIXED", "UNMENTIONED", "UNCLEAR")
AUX_STATES = ("NEGATIVE", "NEUTRAL", "POSITIVE")
AUX_STATE_TO_CLASS = {
    "NEGATIVE": "NEGATIVE", "POSITIVE": "POSITIVE",
    "MIXED": "NEUTRAL", "UNMENTIONED": "NEUTRAL", "UNCLEAR": "NEUTRAL",
}
# NEUTRAL is the pooled human-rubric class; it need not express a neutral attitude.
AUX_OVERALL = ("VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE")
AUX_RELEASE = {
    "protocol_id": "v2_ef356fdf87",
    "label_file_md5": "c22f6c05633214014db47c692e045ea1",
    "split_fingerprint": "038e1d09a2818dfcf6a91dce49e35a22",
    "n_eligible": 102061,
}
AUX_BACKBONES = {"albert": "albert-base-v2", "biobert": "dmis-lab/biobert-base-cased-v1.2"}

# File checks distinguish training implementations and preserve compatible saved runs.
LEGACY_AUX_RUNTIME_SHA256S = ['04dfc14b548f5fa012781902600f5095e2c60a8e330451b8ca0a8d17c5c34c68', '1a2f1322552ba84750fa019d9db4b0891df8747f485adf51b60c12cc28ab6b89']
AUX_RUNTIME_SHA256 = "5347c4ddd749ce9e6b1f344e7745ee09d6d808420c8e3d486c141446b2b75f14"
OUTPUT_PATH = PROJECT_ROOT / EXPERIMENT_FOLDER / "outputs" / AUX_VERSION
ALBERT_MODEL_NAME = AUX_BACKBONES["albert"]
BIOBERT_MODEL_NAME = AUX_BACKBONES["biobert"]
print("Experiment:", EXPERIMENT_FOLDER, "| trials per model:", N_ITERATIONS)
print("Outputs:", OUTPUT_PATH)


Experiment: 03_Full_aspects_aug | trials per model: 10
Outputs: /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class


## 1.4 Mount drive & Load the data

In [8]:
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

def load_label_release(project_root, cfg):
    aux_validate_config(cfg)
    project_root = Path(project_root)
    source = project_root / "00_labelling/output_v2/v2_ef356fdf87/drugscom_plus_ai_labels_v2_ef356fdf87.csv"
    split_path = project_root / "00_split/data/DrugReview_split.csv"
    if aux_hash(source, "md5") != AUX_RELEASE["label_file_md5"]:
        raise ValueError("Label release changed. Restore the pinned release; do not silently relabel.")
    df = pd.read_csv(source, index_col=0).reset_index()
    split = pd.read_csv(split_path)
    required = {"uniqueID", "review", "eligible_both_streams", "rating", "ai_sent5_hard"}
    required |= {f"ai_{a}_state" for a in AUX_ASPECTS}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing release columns: {required - set(df.columns)}")
    if not {"uniqueID", "split", "narrative_group", "is_audit_group"}.issubset(split.columns):
        raise ValueError("Incomplete split manifest.")
    for frame in (df, split):
        if frame.uniqueID.isna().any() or not frame.uniqueID.is_unique:
            raise ValueError("Missing or duplicate review IDs.")
    if set(df.uniqueID) != set(split.uniqueID):
        raise ValueError("Release and manifest ID sets differ.")
    fingerprint = hashlib.md5("|".join(
        f"{u}:{s}" for u, s in sorted(zip(split.uniqueID.astype(str), split["split"]))
    ).encode()).hexdigest()
    if fingerprint != AUX_RELEASE["split_fingerprint"]:
        raise ValueError("Frozen split changed.")
    if not split["split"].isin(["train", "val", "test", "audit"]).all():
        raise ValueError("Unknown split assignment.")
    if split.narrative_group.isna().any() or (split.groupby("narrative_group")["split"].nunique() > 1).any():
        raise ValueError("Missing narrative groups or narratives cross splits.")
    if not split.is_audit_group.isin([True, False]).all() or not (split.is_audit_group == split["split"].eq("audit")).all():
        raise ValueError("Audit reservation is inconsistent.")
    if not df.eligible_both_streams.isin([True, False]).all():
        raise ValueError("Invalid cohort eligibility flags.")
    df = df.loc[df.eligible_both_streams].copy()
    if len(df) != AUX_RELEASE["n_eligible"]:
        raise ValueError("Eligible cohort changed.")
    df = df.drop(columns=[c for c in split if c != "uniqueID" and c in df])
    df = df.merge(split, on="uniqueID", validate="one_to_one").sort_values("uniqueID").reset_index(drop=True)
    return df, split, source, split_path

df, split, source, split_path = load_label_release(PROJECT_ROOT, AUX_CONFIG)
print(f"Loaded {len(df):,} eligible reviews from {source}")


Mounted at /content/drive
Loaded 102,061 eligible reviews from /content/drive/MyDrive/NLP_Projects/03_DrugReview/00_labelling/output_v2/v2_ef356fdf87/drugscom_plus_ai_labels_v2_ef356fdf87.csv


In [9]:
# The loader has checked review IDs, the frozen label file and the saved split.
print("Saved partitions:", df["split"].value_counts().to_dict())


Saved partitions: {'train': 60950, 'test': 20298, 'val': 20256, 'audit': 557}


In [10]:
# Keep the existing review order and split; do not create a new random split.
print("Review IDs are sorted; training seeds do not change partition membership.")


Review IDs are sorted; training seeds do not change partition membership.


# 2. Data Exploration \& Master Data Preparation

## 2.1 Master Data Preparation

### 2.1.1 Text cleaning

In [11]:
def aux_clean_text(text):
    # Same cleaning as the existing hard-label and aspect notebooks.
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

text = df.review
if AUX_CONFIG["augmentation"]:
    # Reuse the exact metadata fields and prefix order from 03/05 _aug.
    for col in ("condition_norm", "ingredient_set_key"):
        df[col] = df[col].fillna("").astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
    df["n_ingredients"] = pd.to_numeric(df.n_ingredients, errors="coerce").fillna(0).astype(int)
    text = df.apply(lambda r: f"[COND] {r.condition_norm} [ING] {r.ingredient_set_key} [N_ING] {r.n_ingredients} [TEXT] {r.review}", axis=1)
df["model_text"] = text.apply(aux_clean_text)

print("Input:", "review plus metadata" if USE_METADATA else "review text only")


Input: review plus metadata


### 2.1.2 Validate & encode aspect labels

NEGATIVE = 0, NEUTRAL = 1, POSITIVE = 2. Source states MIXED, UNMENTIONED and UNCLEAR are grouped into NEUTRAL. The source columns are preserved. The holistic target, when used, keeps its five classes.


In [12]:
# Validate all targets, so baseline and joint conditions have the same cohort.
if not df.ai_sent5_hard.isin(AUX_OVERALL).all():
    raise ValueError("Invalid holistic label; never replace failed labels with neutral.")
rating = pd.to_numeric(df.rating, errors="coerce").round()
if not rating.between(1, 10).all():
    raise ValueError("Invalid rating would change the common cohort.")
df["original_id"] = ((rating.astype(int) - 1) // 2).astype(int)
df["holistic_id"] = df.ai_sent5_hard.map({x: i for i, x in enumerate(AUX_OVERALL)}).astype(int)
for aspect in AUX_ASPECTS:
    col = f"ai_{aspect}_state"
    if not df[col].isin(AUX_SOURCE_STATES).all():
        raise ValueError(f"Invalid five-state label: {col}")
    df[f"{aspect}_id"] = df[col].map(AUX_STATE_TO_CLASS).map(
        {x: i for i, x in enumerate(AUX_STATES)}).astype(int)

for aspect in ASPECT_NAMES:
    print(aspect, df[f"{aspect}_id"].value_counts().sort_index().to_dict())


efficacy {0: 19761, 1: 17551, 2: 64749}
safety {0: 51921, 1: 34895, 2: 15245}
burden {0: 40844, 1: 51341, 2: 9876}
cost {0: 4689, 1: 94905, 2: 2467}


## 2.3 Master Data Split

Use the existing train, validation, test and reserved audit assignments from `00_split`. Training and parameter selection use only train and validation rows.


In [13]:
frames = {s: df.loc[df["split"].eq(s)].copy().reset_index(drop=True) for s in ("train", "val", "test", "audit")}
if any(len(frame) == 0 for frame in frames.values()):
    raise ValueError("Each frozen partition must be nonempty.")
identity = {
    "release": AUX_RELEASE, "split_file_sha256": aux_hash(split_path),
    "cohort_sha256": hashlib.sha256("|".join(df.uniqueID.astype(str)).encode()).hexdigest(),
    "split_counts": {s: len(f) for s, f in frames.items()},
    "text_sha256": hashlib.sha256("\n".join(df.model_text).encode()).hexdigest(),
}

print("Partitions:", identity["split_counts"])


Partitions: {'train': 60950, 'val': 20256, 'test': 20298, 'audit': 557}


# 3. ALBERT \& BioBERT Models (Multi-aspect)

## 3.1 Shared helper functions

These functions calculate losses and evaluation scores and write prediction files. A model output layer is called a “head”: each aspect head predicts three classes, and the overall head predicts five. Model probabilities come from the trained classifier, not from the GPT annotation API.


In [14]:
def aux_task_labels(frame, cfg):
    labels = {a: frame[f"{a}_id"].to_numpy() for a in cfg["aspects"]}
    if cfg["target"] != "aspects":
        labels["overall"] = frame[f"{cfg['target']}_id"].to_numpy()
    return labels

def aux_criteria(train_labels, device):
    # Retain balanced CE within each task, calculated on TRAIN only.
    criteria, weights = {}, {}
    for task, values in train_labels.items():
        n_classes = len(AUX_OVERALL) if task == "overall" else len(AUX_STATES)
        values = np.asarray(values)
        if values.ndim != 1 or len(values) == 0 or not np.issubdtype(values.dtype, np.integer) or not ((values >= 0) & (values < n_classes)).all():
            raise ValueError(f"Invalid {n_classes}-class training targets: {task}")
        counts = np.bincount(values, minlength=n_classes)
        seen = counts > 0
        weight = np.zeros(n_classes, dtype=np.float32)
        weight[seen] = len(values) / (seen.sum() * counts[seen])
        weights[task] = weight.tolist()
        criteria[task] = nn.CrossEntropyLoss(weight=torch.tensor(weight, device=device))
    return criteria, weights

def aux_loss(logits, targets, criteria, strength):
    terms = {k: criteria[k](logits[k].float(), targets[k]) for k in logits}
    aspect_terms = [v for k, v in terms.items() if k != "overall"]
    if "overall" not in terms:
        return torch.stack(aspect_terms).mean(), terms
    total = terms["overall"]
    if aspect_terms:
        total = total + strength * torch.stack(aspect_terms).mean()
    return total, terms

def aux_metrics(y, pred, ordinal=False):
    # Aspect macro F1 averages three classes; overall metrics retain five.
    class_ids = range(len(AUX_OVERALL) if ordinal else len(AUX_STATES))
    out = {
        "n": len(y), "accuracy": float(accuracy_score(y, pred)),
        "weighted_f1": float(f1_score(y, pred, labels=class_ids, average="weighted", zero_division=0)),
        "macro_f1": float(f1_score(y, pred, labels=class_ids, average="macro", zero_division=0)),
    }
    if ordinal:
        qwk = float(cohen_kappa_score(y, pred, labels=range(5), weights="quadratic"))
        out.update(mae=float(np.mean(np.abs(np.asarray(y) - pred))), qwk=qwk if np.isfinite(qwk) else None)
    return out

def aux_predict(model, loader, device):
    model.eval()
    probs, labels = {}, {}
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            for task, values in logits.items():
                require_finite_tensor(values, f"{task} prediction logits")
                probabilities = values.float().softmax(-1).cpu().numpy()
                require_probabilities(probabilities, values.shape[-1])
                probs.setdefault(task, []).append(probabilities)
                labels.setdefault(task, []).append(batch["targets"][task].numpy())
    return {k: np.concatenate(v) for k, v in probs.items()}, {k: np.concatenate(v) for k, v in labels.items()}

def aux_selection(probs, labels):
    for task, p in probs.items():
        require_probabilities(p, 5 if task == "overall" else 3)
    if "overall" in probs:
        return float(f1_score(labels["overall"], probs["overall"].argmax(1), average="weighted", zero_division=0))
    return float(np.mean([f1_score(labels[k], p.argmax(1), average="weighted", zero_division=0) for k, p in probs.items()]))

def aux_trial_plan(cfg):
    rng = np.random.RandomState(cfg["seed"])
    return [{"lr": float(10**rng.uniform(np.log10(1e-5), np.log10(3e-5))),
             "epochs": int(rng.randint(4, 10)), "dropout": float(rng.uniform(0.0, 0.25)),
             "seed": cfg["seed"] + i} for i in range(cfg["n_trials"])]

def aux_export(model, loader, frame, cfg, model_name, split_name, output, device):
    probs, labels = aux_predict(model, loader, device)
    audit_columns = [c for c in ("is_human_annotated", "is_audit_group", "audit_stratum") if c in frame]
    result = frame[["uniqueID", "narrative_group", "split"] + audit_columns].copy()
    metrics = []
    for task, p in probs.items():
        names = AUX_OVERALL if task == "overall" else AUX_STATES
        if p.ndim != 2 or p.shape != (len(frame), len(names)):
            raise ValueError(f"Unexpected probability shape for {task}: {p.shape}")
        require_probabilities(p, len(names))
        pred, y = p.argmax(1), labels[task]
        result[f"{task}_true_id"] = y
        result[f"{task}_pred_id"] = pred
        result[f"{task}_pred"] = np.asarray(names)[pred]
        for k, name in enumerate(names):
            result[f"{task}_p_{name}"] = p[:, k]
        row = {"model": model_name, "task": task, "partition": split_name, "reference_source": "rating" if cfg["target"] == "original" and task == "overall" else "GPT-4o-mini", "human_agreement": False, **aux_metrics(y, pred, task == "overall")}
        metrics.append(row)
        if split_name == "test":
            pd.DataFrame(classification_report(y, pred, labels=range(len(names)), target_names=names,
                output_dict=True, zero_division=0)).T.to_csv(output / f"{model_name}_{task}_class_report.csv")
            pd.DataFrame(confusion_matrix(y, pred, labels=range(len(names))), index=names, columns=names).to_csv(
                output / f"{model_name}_{task}_confusion.csv")
        if task != "overall":
            # Native three-class predictions; these compatibility columns are exact aliases.
            result[f"{task}_true_3class_id"] = y
            result[f"{task}_pred_3class_id"] = pred
            source_col = f"ai_{task}_state"
            if source_col in frame:
                result[f"{task}_source_state"] = frame[source_col].to_numpy()
                result[f"{task}_is_mentioned"] = frame[source_col].ne("UNMENTIONED").to_numpy()
    result.to_csv(output / f"{model_name}_{split_name}_predictions.csv", index=False)
    return metrics

def prepare_experiment(project_root, cfg, frames, identity):
    """Check saved settings and prepare the output folder and training losses."""
    import transformers
    import sklearn
    output = Path(project_root) / cfg["folder"] / "outputs" / AUX_VERSION
    output.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    plan = aux_trial_plan(cfg)
    comparable_cfg = {k: v for k, v in cfg.items() if k not in ("folder", "backbones", "notebook")}
    signature = {"version": AUX_VERSION, "runtime_sha256": AUX_RUNTIME_SHA256,
        "config": comparable_cfg, "data": identity, "trials": plan,
        "aspect_schema": {"classes": list(AUX_STATES), "source_mapping": AUX_STATE_TO_CLASS},
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "sklearn": sklearn.__version__}
    # Never silently load stale checkpoints when data, settings, code or libraries differ.
    identity_path = output / "experiment_identity.json"
    if identity_path.exists():
        saved_signature = json.loads(identity_path.read_text())
        expected_signature = dict(signature)
        # Accept the previous cell layout only when every data/training setting matches.
        # Keep the original saved identity; do not relabel an old run as a new one.
        if saved_signature.get("runtime_sha256") in LEGACY_AUX_RUNTIME_SHA256S:
            if not all((output / model_name / "complete.json").exists() for model_name in cfg["backbones"]):
                raise ValueError("This unfinished run uses earlier training code. Keep its original notebook to resume it; do not mix implementations in one run.")
            expected_signature["runtime_sha256"] = saved_signature["runtime_sha256"]
        if saved_signature != expected_signature:
            raise ValueError("Existing outputs belong to a different configuration. Use a new output version/folder.")
        signature = saved_signature
    aux_json(identity_path, signature)
    aux_json(output / "run_environment.json", {"device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "notebook_code_sha256": AUX_RUNTIME_SHA256})
    for name in ("train", "val", "test", "audit"):
        frames[name][["uniqueID", "split", "narrative_group"]].to_csv(output / f"{name}_ids.csv", index=False)
    print("Partitions:", identity["split_counts"], "| target:", cfg["target"], "| aspects:", cfg["aspects"])
    train_labels = aux_task_labels(frames["train"], cfg)
    criteria, class_weights = aux_criteria(train_labels, device)
    aux_json(output / "train_class_weights.json", class_weights)
    return output, device, plan, criteria, class_weights


## 3.2 Data splits \& labels

In [15]:
train_labels = aux_task_labels(frames["train"], AUX_CONFIG)
val_labels = aux_task_labels(frames["val"], AUX_CONFIG)
test_labels = aux_task_labels(frames["test"], AUX_CONFIG)

output, device, plan, criteria, class_weights = prepare_experiment(
    PROJECT_ROOT, AUX_CONFIG, frames, identity
)
print("Parameter search:")
print(pd.DataFrame(plan).to_string(index=False))
print("Class weights from training data:", class_weights)


Partitions: {'train': 60950, 'val': 20256, 'test': 20298, 'audit': 557} | target: aspects | aspects: ['efficacy', 'safety', 'burden', 'cost']
Parameter search:
      lr  epochs  dropout  seed
0.000012       6 0.085351  2025
0.000011       7 0.097059  2026
0.000013       9 0.195692  2027
0.000011       9 0.200246  2028
0.000016       4 0.094601  2029
0.000018       5 0.000793  2030
0.000014       9 0.183296  2031
0.000011       7 0.062150  2032
0.000021       6 0.155208  2033
0.000030       9 0.236536  2034
Class weights from training data: {'efficacy': [1.7274608612060547, 1.9261155128479004, 0.5257800221443176], 'safety': [0.653416097164154, 0.9769037365913391, 2.242457628250122], 'burden': [0.8348743319511414, 0.6606616377830505, 3.4652340412139893], 'cost': [7.353118419647217, 0.3579587936401367, 14.207459449768066]}


## 3.3 Generic multi-task model componets

Full-aspects uses the mean of the four aspect cross-entropy losses. Joint models use overall cross-entropy plus 1.0 times the mean of their selected aspect losses. Each task uses training-derived class weights.

Select models by validation overall weighted F1, or by the mean of the four aspect weighted F1 scores for Full-aspects. The current training calculation is retained; the reference notebook's different loss is not copied.


In [16]:
def verify_completed_numerics(model_output):
    for prediction_path in model_output.glob("*_predictions.csv"):
        saved = pd.read_csv(prediction_path)
        for task in ["overall", *AUX_ASPECTS]:
            names = AUX_OVERALL if task == "overall" else AUX_STATES
            columns = [f"{task}_p_{name}" for name in names]
            if all(c in saved for c in columns):
                require_probabilities(saved[columns].to_numpy(), len(names))
    payload = torch.load(model_output / "best_model.pth", map_location="cpu", weights_only=True)
    for name, value in payload["state_dict"].items():
        if value.is_floating_point():
            require_finite_tensor(value, f"saved model state {name}")

class AuxiliaryDataset(Dataset):
    """All labels are supervision only; model.forward accepts text tensors only."""
    def __init__(self, encoded, labels):
        self.encoded = encoded
        self.labels = {k: torch.as_tensor(v, dtype=torch.long) for k, v in labels.items()}

    def __len__(self):
        return len(self.encoded["input_ids"])

    def __getitem__(self, index):
        return {"input_ids": self.encoded["input_ids"][index],
                "attention_mask": self.encoded["attention_mask"][index],
                "targets": {k: v[index] for k, v in self.labels.items()}}

class AuxiliaryClassifier(nn.Module):
    """Existing shared CLS encoder with a primary head and selected aspect heads."""
    def __init__(self, base, aspects, primary=True, dropout=0.1):
        super().__init__()
        self.base = base
        self.dropout = nn.Dropout(dropout)
        # Initialize primary head first, so H and H+aspects start it identically.
        self.primary_head = nn.Linear(base.config.hidden_size, 5) if primary else None
        self.aspect_heads = nn.ModuleDict({a: nn.Linear(base.config.hidden_size, len(AUX_STATES)) for a in aspects})

    def forward(self, input_ids, attention_mask):
        cls = self.dropout(self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :])
        result = {a: head(cls) for a, head in self.aspect_heads.items()}
        if self.primary_head is not None:
            result["overall"] = self.primary_head(cls)
        return result

def aux_fit(model, train_loader, val_loader, criteria, hp, cfg, device):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=hp["lr"],
        total_steps=hp["epochs"] * len(train_loader), pct_start=0.1)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and dtype == torch.float16)
    best, state, best_epoch, stalled = -1.0, None, -1, 0
    history = []
    for epoch in range(1, hp["epochs"] + 1):
        model.train()
        losses, per_task = [], {k: [] for k in criteria}
        for batch in train_loader:
            targets = {k: v.to(device) for k, v in batch["targets"].items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
                logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
                loss, terms = aux_loss(logits, targets, criteria, cfg["aux_strength"])
            checked_optimizer_step(loss, model, optimizer, scaler, 1.0, scheduler)
            losses.append(loss.item())
            for task, value in terms.items():
                per_task[task].append(value.item())
        probs, labels = aux_predict(model, val_loader, device)
        score = aux_selection(probs, labels)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "selection_f1": score,
               "task_losses": {k: float(np.mean(v)) for k, v in per_task.items()},
               "validation": {k: aux_metrics(labels[k], p.argmax(1), k == "overall") for k, p in probs.items()}}
        history.append(row)
        print(f"Epoch {epoch}/{hp['epochs']} | loss={row['train_loss']:.4f} | validation selection F1={score:.4f}", flush=True)
        if score > best:
            require_finite_model(model)
            best, best_epoch, stalled = score, epoch, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stalled += 1
            if stalled >= cfg["patience"]:
                break
    if state is None:
        raise RuntimeError("No valid checkpoint.")
    model.load_state_dict(state)
    return best, best_epoch, history

def prepare_model(model_name, frames, cfg, output, device):
    """Tokenize this model's training/validation text, or verify its completed run."""
    from transformers import AutoTokenizer
    if model_name not in cfg["backbones"]:
        print(model_name, "is not selected in BACKBONES; skipping.")
        return None
    model_output = output / model_name
    model_output.mkdir(exist_ok=True)
    done = model_output / "complete.json"
    run = {"name": model_name, "output": model_output, "done": done,
           "complete": done.exists()}
    if done.exists():
        complete = json.loads(done.read_text())
        for filename, digest in complete["artifacts"].items():
            if not (model_output / filename).exists() or aux_hash(model_output / filename) != digest:
                raise ValueError(f"Completed artifact changed: {model_name}/{filename}")
        verify_completed_numerics(model_output)
        print(model_name, "already complete and verified; skipping.")
        return run
    tokenizer = AutoTokenizer.from_pretrained(AUX_BACKBONES[model_name])
    datasets = {}
    # Test/audit predictions are made after selecting the model on validation data.
    for partition in ("train", "val"):
        encoded = tokenizer(frames[partition].model_text.tolist(), truncation=True,
            padding="max_length", max_length=cfg["max_length"], return_tensors="pt")
        datasets[partition] = AuxiliaryDataset(encoded, aux_task_labels(frames[partition], cfg))
    loader_args = {"batch_size": cfg["batch_size"], "num_workers": cfg["num_workers"],
                   "pin_memory": device.type == "cuda"}
    val_loader = DataLoader(datasets["val"], shuffle=False,
        generator=torch.Generator().manual_seed(cfg["seed"] + 100_000), **loader_args)
    run.update(tokenizer=tokenizer, datasets=datasets, loader_args=loader_args,
               val_loader=val_loader)
    return run

def evaluate_saved_model(run, frames, cfg, best_record, device):
    """Load the selected model, evaluate each partition and save the existing outputs."""
    from transformers import AutoModel
    if run is None:
        return pd.DataFrame(columns=["model", "task", "partition"])
    if run["complete"]:
        complete = json.loads(run["done"].read_text())
        for filename, digest in complete["artifacts"].items():
            if not (run["output"] / filename).exists() or aux_hash(run["output"] / filename) != digest:
                raise ValueError(f"Completed artifact changed: {run['name']}/{filename}")
        verify_completed_numerics(run["output"])
        return pd.read_csv(run["output"] / "metrics.csv")
    model_name, model_output, done = run["name"], run["output"], run["done"]
    tokenizer, datasets = run["tokenizer"], run["datasets"]
    loader_args = run["loader_args"]
    payload = torch.load(model_output / "best_model.pth", map_location="cpu", weights_only=True)
    if payload["record"] != best_record:
        raise ValueError("Selected checkpoint does not match completed trial records.")
    model = AuxiliaryClassifier(AutoModel.from_pretrained(AUX_BACKBONES[model_name]),
        cfg["aspects"], primary=cfg["target"] != "aspects", dropout=best_record["hp"]["dropout"])
    model.load_state_dict(payload["state_dict"], strict=True)
    require_finite_model(model)
    model.to(device)
    rows = []
    for partition in ("val", "test", "audit"):
        if partition not in datasets:
            encoded = tokenizer(frames[partition].model_text.tolist(), truncation=True,
                padding="max_length", max_length=cfg["max_length"], return_tensors="pt")
            datasets[partition] = AuxiliaryDataset(encoded, aux_task_labels(frames[partition], cfg))
        loader = DataLoader(datasets[partition], shuffle=False,
            generator=torch.Generator().manual_seed(cfg["seed"] + 200_000), **loader_args)
        rows += aux_export(model, loader, frames[partition], cfg, model_name, partition, model_output, device)
    pd.DataFrame(rows).to_csv(model_output / "metrics.csv", index=False)
    tokenizer.save_pretrained(model_output / "tokenizer")
    model.base.config.save_pretrained(model_output / "encoder_config")
    aux_json(model_output / "selected_parameters.json", best_record)
    print(pd.DataFrame(rows).query("partition == 'test'").to_string(index=False))
    artifacts = {p.relative_to(model_output).as_posix(): aux_hash(p)
                 for p in model_output.rglob("*") if p.is_file() and p != done}
    aux_json(done, {"artifacts": artifacts})
    del model, loader
    run.pop("datasets", None)
    run.pop("val_loader", None)
    run["complete"] = True
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


## 3.4 ALBERT: Multi-task training

The following three steps prepare text, search the existing parameter range and evaluate the selected model. Completed models and trials are reused as before; restarting Colab alone does not force retraining.


In [17]:
print("\n" + "=" * 60)
print('ALBERT' + " — DrugReview multi-task training")
print("=" * 60)



ALBERT — DrugReview multi-task training


### 3.4.1 Pre-tokenize

In [18]:
albert_run = prepare_model('albert', frames, AUX_CONFIG, output, device)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### 3.4.2 Hyperparameter tuning

In [19]:
best_record_albert = None
if albert_run is not None and not albert_run["complete"]:
    for index, hp in enumerate(plan):
        trial_path = albert_run["output"] / f"trial_{index + 1}.json"
        # A saved completed trial can be reused on interruption; the selected checkpoint is always updated first.
        if trial_path.exists():
            record = json.loads(trial_path.read_text())
            if record["hp"] != hp:
                raise ValueError("Saved trial plan differs.")
            if best_record_albert is None or record["score"] > best_record_albert["score"]:
                best_record_albert = record
            continue
        aux_seed(hp["seed"])
        model = AuxiliaryClassifier(AutoModel.from_pretrained(AUX_BACKBONES['albert']),
            AUX_CONFIG["aspects"], primary=AUX_CONFIG["target"] != "aspects", dropout=hp["dropout"])
        # Reset after head construction: identical data order and dropout RNG across paired conditions.
        aux_seed(hp["seed"])
        train_loader = DataLoader(albert_run["datasets"]["train"], shuffle=True,
            generator=torch.Generator().manual_seed(hp["seed"]), worker_init_fn=aux_worker_seed, **albert_run["loader_args"])
        print(f"albert trial {index + 1}/{len(plan)}", hp, flush=True)
        score, epoch, history = aux_fit(model, train_loader, albert_run["val_loader"], criteria, hp, AUX_CONFIG, device)
        record = {"trial": index + 1, "hp": hp, "score": score, "selected_epoch": epoch}
        if best_record_albert is None or score > best_record_albert["score"]:
            best_record_albert = record
            tmp = albert_run["output"] / "best_model.tmp"
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "record": record}, tmp)
            tmp.replace(albert_run["output"] / "best_model.pth")
        aux_json(albert_run["output"] / f"trial_{index + 1}_history.json", history)
        aux_json(trial_path, record)
        del model, train_loader
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

albert trial 1/10 {'lr': 1.160497696239576e-05, 'epochs': 6, 'dropout': 0.08535068283342875, 'seed': 2025}


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/6 | loss=0.9134 | validation selection F1=0.7252
Epoch 2/6 | loss=0.6198 | validation selection F1=0.7716
Epoch 3/6 | loss=0.5297 | validation selection F1=0.7784
Epoch 4/6 | loss=0.4667 | validation selection F1=0.7956
Epoch 5/6 | loss=0.4226 | validation selection F1=0.7980
Epoch 6/6 | loss=0.3985 | validation selection F1=0.7990
albert trial 2/10 {'lr': 1.1229524450717097e-05, 'epochs': 7, 'dropout': 0.09705888652849565, 'seed': 2026}
Epoch 1/7 | loss=0.9176 | validation selection F1=0.7547
Epoch 2/7 | loss=0.6190 | validation selection F1=0.7790
Epoch 3/7 | loss=0.5284 | validation selection F1=0.7944
Epoch 4/7 | loss=0.4665 | validation selection F1=0.8044
Epoch 5/7 | loss=0.4170 | validation selection F1=0.7996
Epoch 6/7 | loss=0.3796 | validation selection F1=0.8012
Epoch 7/7 | loss=0.3621 | validation selection F1=0.8028
albert trial 3/10 {'lr': 1.3271033162649868e-05, 'epochs': 9, 'dropout': 0.195692122900145, 'seed': 2027}
Epoch 1/9 | loss=0.9685 | validation selectio

### 3.4.3 Model evaluation

In [20]:
albert_results = evaluate_saved_model(
    albert_run, frames, AUX_CONFIG, best_record_albert, device
)
print(albert_results.query("partition == 'test'").to_string(index=False))


 model     task partition reference_source  human_agreement     n  accuracy  weighted_f1  macro_f1
albert efficacy      test      GPT-4o-mini            False 20298  0.783427     0.790906  0.713003
albert   safety      test      GPT-4o-mini            False 20298  0.797172     0.797436  0.784197
albert   burden      test      GPT-4o-mini            False 20298  0.694453     0.696994  0.635353
albert     cost      test      GPT-4o-mini            False 20298  0.974924     0.975921  0.864321
 model     task partition reference_source  human_agreement     n  accuracy  weighted_f1  macro_f1
albert efficacy      test      GPT-4o-mini            False 20298  0.783427     0.790906  0.713003
albert   safety      test      GPT-4o-mini            False 20298  0.797172     0.797436  0.784197
albert   burden      test      GPT-4o-mini            False 20298  0.694453     0.696994  0.635353
albert     cost      test      GPT-4o-mini            False 20298  0.974924     0.975921  0.864321


## 3.5 BioBERT: Multi-task training

The following three steps prepare text, search the existing parameter range and evaluate the selected model. Completed models and trials are reused as before; restarting Colab alone does not force retraining.


In [21]:
print("\n" + "=" * 60)
print('BioBERT' + " — DrugReview multi-task training")
print("=" * 60)



BioBERT — DrugReview multi-task training


### 3.5.1 Pre-tokenize

In [22]:
biobert_run = prepare_model('biobert', frames, AUX_CONFIG, output, device)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

### 3.5.2 Hyperparameter tuning

In [23]:
best_record_biobert = None
if biobert_run is not None and not biobert_run["complete"]:
    for index, hp in enumerate(plan):
        trial_path = biobert_run["output"] / f"trial_{index + 1}.json"
        # A saved completed trial can be reused on interruption; the selected checkpoint is always updated first.
        if trial_path.exists():
            record = json.loads(trial_path.read_text())
            if record["hp"] != hp:
                raise ValueError("Saved trial plan differs.")
            if best_record_biobert is None or record["score"] > best_record_biobert["score"]:
                best_record_biobert = record
            continue
        aux_seed(hp["seed"])
        model = AuxiliaryClassifier(AutoModel.from_pretrained(AUX_BACKBONES['biobert']),
            AUX_CONFIG["aspects"], primary=AUX_CONFIG["target"] != "aspects", dropout=hp["dropout"])
        # Reset after head construction: identical data order and dropout RNG across paired conditions.
        aux_seed(hp["seed"])
        train_loader = DataLoader(biobert_run["datasets"]["train"], shuffle=True,
            generator=torch.Generator().manual_seed(hp["seed"]), worker_init_fn=aux_worker_seed, **biobert_run["loader_args"])
        print(f"biobert trial {index + 1}/{len(plan)}", hp, flush=True)
        score, epoch, history = aux_fit(model, train_loader, biobert_run["val_loader"], criteria, hp, AUX_CONFIG, device)
        record = {"trial": index + 1, "hp": hp, "score": score, "selected_epoch": epoch}
        if best_record_biobert is None or score > best_record_biobert["score"]:
            best_record_biobert = record
            tmp = biobert_run["output"] / "best_model.tmp"
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "record": record}, tmp)
            tmp.replace(biobert_run["output"] / "best_model.pth")
        aux_json(biobert_run["output"] / f"trial_{index + 1}_history.json", history)
        aux_json(trial_path, record)
        del model, train_loader
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

biobert trial 1/10 {'lr': 1.160497696239576e-05, 'epochs': 6, 'dropout': 0.08535068283342875, 'seed': 2025}


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/6 | loss=0.9051 | validation selection F1=0.7264
Epoch 2/6 | loss=0.6444 | validation selection F1=0.7616
Epoch 3/6 | loss=0.5578 | validation selection F1=0.7687
Epoch 4/6 | loss=0.5066 | validation selection F1=0.7828
Epoch 5/6 | loss=0.4769 | validation selection F1=0.7893
Epoch 6/6 | loss=0.4635 | validation selection F1=0.7899
biobert trial 2/10 {'lr': 1.1229524450717097e-05, 'epochs': 7, 'dropout': 0.09705888652849565, 'seed': 2026}
Epoch 1/7 | loss=0.9305 | validation selection F1=0.7324
Epoch 2/7 | loss=0.6535 | validation selection F1=0.7697
Epoch 3/7 | loss=0.5628 | validation selection F1=0.7833
Epoch 4/7 | loss=0.5076 | validation selection F1=0.7923
Epoch 5/7 | loss=0.4728 | validation selection F1=0.7911
Epoch 6/7 | loss=0.4506 | validation selection F1=0.7927
Epoch 7/7 | loss=0.4422 | validation selection F1=0.7937
biobert trial 3/10 {'lr': 1.3271033162649868e-05, 'epochs': 9, 'dropout': 0.195692122900145, 'seed': 2027}
Epoch 1/9 | loss=0.9451 | validation select

### 3.5.3 Model Evaluation

In [24]:
biobert_results = evaluate_saved_model(
    biobert_run, frames, AUX_CONFIG, best_record_biobert, device
)
print(biobert_results.query("partition == 'test'").to_string(index=False))


  model     task partition reference_source  human_agreement     n  accuracy  weighted_f1  macro_f1
biobert efficacy      test      GPT-4o-mini            False 20298  0.776234     0.787375  0.712221
biobert   safety      test      GPT-4o-mini            False 20298  0.795054     0.795748  0.783045
biobert   burden      test      GPT-4o-mini            False 20298  0.686964     0.692804  0.635122
biobert     cost      test      GPT-4o-mini            False 20298  0.971672     0.973144  0.848355
  model     task partition reference_source  human_agreement     n  accuracy  weighted_f1  macro_f1
biobert efficacy      test      GPT-4o-mini            False 20298  0.776234     0.787375  0.712221
biobert   safety      test      GPT-4o-mini            False 20298  0.795054     0.795748  0.783045
biobert   burden      test      GPT-4o-mini            False 20298  0.686964     0.692804  0.635122
biobert     cost      test      GPT-4o-mini            False 20298  0.971672     0.973144  0.848355


## 3.6 Save Prediction

The evaluation cells save per-review predictions and model files in the existing `albert/` and `biobert/` output folders. This last cell saves a combined metrics summary and lists the prediction files. Test and human-audit predictions remain separate.


In [25]:
all_model_metrics = pd.concat([albert_results, biobert_results], ignore_index=True)
summary_path = output / "all_models_metrics.csv"
all_model_metrics.to_csv(summary_path, index=False)
print(f"Saved model summary → {summary_path}")
for model_name in AUX_CONFIG["backbones"]:
    for partition in ("val", "test", "audit"):
        path = output / model_name / f"{model_name}_{partition}_predictions.csv"
        print(f"{model_name} {partition} predictions → {path}")
print(all_model_metrics.query("partition == 'test'").to_string(index=False))


Saved model summary → /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class/all_models_metrics.csv
albert val predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class/albert/albert_val_predictions.csv
albert test predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class/albert/albert_test_predictions.csv
albert audit predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class/albert/albert_audit_predictions.csv
biobert val predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class/biobert/biobert_val_predictions.csv
biobert test predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/03_Full_aspects_aug/outputs/aspect_auxiliary_v2_3class/biobert/biobert_test_predictions.csv
biobert audit pr